# Checkpoint Lineage Walkthrough (User-Facing)

This notebook is a complete tour of the checkpoint model.

What we cover:
- Auto-generated `workflow_id`
- Strict resume rules for same `workflow_id`
- Explicit branching via `checkpoint = cp.checkpoint(...)`
- Failed-run resume on same `workflow_id` (implicit retry)
- `graph_struct_hash` vs `graph_code_hash`
- Gate decision values in state
- Nested graph lineage and child run IDs
- Git-like lineage visualization with `cp.lineage(workflow_id)`
- PAUSED resume with interrupt (the one section that needs `await`)


## Feature Map

| Feature | Main API | Where in this notebook |
|---|---|---|
| Auto IDs | `runner.run(graph, values)` with checkpointer | Section 1 |
| Strict resume | Same `workflow_id` without overrides | Section 2 |
| Explicit fork | `cp.checkpoint(...)` + `runner.run(..., checkpoint=..., workflow_id=...)` | Section 3 |
| Failed resume | rerun same `workflow_id` after fixing issue | Section 4 |
| Hashes | `graph.structural_hash`, `graph.code_hash` | Section 5 |
| Gate values persisted | `cp.state(workflow_id)` includes `_gate_name` | Section 6 |
| Nested lineage | child run ids like `root/nested` + `cp.lineage(...)` | Section 7 |
| Interrupt resume | `await AsyncRunner.run(...)` | Section 8 |


In [ ]:
from __future__ import annotations

from pathlib import Path

from hypergraph import END, AsyncRunner, Graph, RunStatus, SyncRunner, ifelse, interrupt, node
from hypergraph.checkpointers import SqliteCheckpointer
from hypergraph.exceptions import (
    GraphChangedError,
    InputOverrideRequiresForkError,
    WorkflowAlreadyCompletedError,
)

DB_PATH = Path('./tmp/checkpoint-lineage-demo.db')
DB_PATH.parent.mkdir(parents=True, exist_ok=True)
for p in [DB_PATH, Path(str(DB_PATH) + '-shm'), Path(str(DB_PATH) + '-wal')]:
    p.unlink(missing_ok=True)

cp = SqliteCheckpointer(str(DB_PATH))
runner = SyncRunner(checkpointer=cp)

print(f'Using sqlite DB: {DB_PATH.resolve()}')


## 1) Auto workflow IDs + checkpoint metadata

If a checkpointer exists and you do **not** pass `workflow_id`, Hypergraph generates one.


In [ ]:
@node(output_name='doubled')
def double(x: int) -> int:
    return x * 2

auto_graph = Graph([double])
auto_result = runner.run(auto_graph, {'x': 5})
auto_run = cp.get_run(auto_result.workflow_id)

print('workflow_id:', auto_result.workflow_id)
print('output doubled:', auto_result['doubled'])
print('status:', auto_run.status.value)
print('config keys:', sorted(auto_run.config.keys()))


## 2) Strict resume rules on the same `workflow_id`

Same `workflow_id` now means **same lineage**:
- COMPLETED is terminal
- overriding values requires fork
- structural graph change requires fork


In [ ]:
@node(output_name='tripled')
def triple(doubled: int) -> int:
    return doubled * 3

lineage_graph = Graph([double, triple])
runner.run(lineage_graph, {'x': 3}, workflow_id='wf-lineage')

try:
    runner.run(lineage_graph, workflow_id='wf-lineage')
except WorkflowAlreadyCompletedError as e:
    print('completed guard:', type(e).__name__)

try:
    runner.run(lineage_graph, {'x': 99}, workflow_id='wf-lineage')
except InputOverrideRequiresForkError as e:
    print('override guard:', type(e).__name__)

# Structural change guard.
changed_graph = Graph([double])
runner.run(changed_graph, {'x': 2}, workflow_id='wf-graph-v1')
try:
    runner.run(lineage_graph, workflow_id='wf-graph-v1')
except GraphChangedError as e:
    print('graph-change guard:', type(e).__name__)


## 3) Explicit fork using a checkpoint

User-facing flow:
1. Get checkpoint by workflow id (optionally at a superstep)
2. Run with `checkpoint=...` and a **new** `workflow_id`


In [ ]:
root_checkpoint = cp.checkpoint('wf-lineage')
print('checkpoint has values keys:', sorted(root_checkpoint.values.keys()))

fork_result = runner.run(lineage_graph, {'x': 100}, checkpoint=root_checkpoint, workflow_id='wf-lineage-fork')
fork_run = cp.get_run(fork_result.workflow_id)

print('fork id:', fork_result.workflow_id)
print('fork tripled:', fork_result['tripled'])
print('fork metadata:', {'forked_from': fork_run.forked_from, 'fork_superstep': fork_run.fork_superstep})

# Visualize lineage lanes
cp.lineage('wf-lineage-fork')


## 4) Failed-run resume (implicit retry on same workflow_id)

For FAILED workflows, fix the issue and rerun the same `workflow_id`.


In [ ]:
fail_switch = {'on': True}

@node(output_name='seed')
def seed(x: int) -> int:
    return x

@node(output_name='out')
def flaky(seed: int) -> int:
    if fail_switch['on']:
        raise RuntimeError('transient failure')
    return seed * 10

failed_graph = Graph([seed, flaky])
first = runner.run(failed_graph, {'x': 7}, workflow_id='wf-failed', error_handling='continue')
print('first status:', first.status.value)

# Same-ID resume (no new values)
fail_switch['on'] = False
resumed = runner.run(failed_graph, workflow_id='wf-failed', on_internal_override='ignore')
print('resumed id:', resumed.workflow_id)
print('resumed status:', resumed.status.value, '| out =', resumed['out'])


## 5) `graph_struct_hash` vs `graph_code_hash`

- `graph_struct_hash`: compatibility for same-ID resume
- `graph_code_hash`: code-sensitive signal for caching/observability

This example keeps structure identical and only changes function code.


In [ ]:
@node(output_name='y')
def transform(x: int) -> int:
    return x + 1

g1 = Graph([transform])

@node(output_name='y')
def transform(x: int) -> int:
    return x + 100

g2 = Graph([transform])

print('structural_hash equal:', g1.structural_hash == g2.structural_hash)
print('code_hash equal:', g1.code_hash == g2.code_hash)
print('structural hashes:', g1.structural_hash[:10], g2.structural_hash[:10])
print('code hashes:', g1.code_hash[:10], g2.code_hash[:10])


## 6) Gate values are persisted in checkpoint state

Gate decisions are stored as internal values (`_gate_name`) so route intent is recoverable.


In [ ]:
@node(output_name='next_count')
def increment(count: int) -> int:
    return count + 1

@ifelse(when_true=END, when_false='increment')
def done(next_count: int) -> bool:
    return next_count >= 2

gate_graph = Graph([increment, done])
runner.run(gate_graph, {'count': 2}, workflow_id='wf-gate')

state = cp.state('wf-gate')
print('contains _done:', '_done' in state)
print('_done value:', state.get('_done'))


## 7) Nested graphs are first-class in lineage

- child runs are materialized (`parent/nested_node`)
- structural changes inside nested graphs are guarded
- forking nested workflows preserves lineage


In [ ]:
@node(output_name='inner_out')
def inner_v1(x: int) -> int:
    return x + 1

@node(output_name='inner_mid')
def inner_v2_mid(x: int) -> int:
    return x + 2

@node(output_name='inner_out')
def inner_v2_end(inner_mid: int) -> int:
    return inner_mid * 2

@node(output_name='final_nested')
def finish_nested(inner_out: int) -> int:
    return inner_out + 10

outer_v1 = Graph([Graph([inner_v1], name='inner').as_node(name='nested'), finish_nested])
outer_v2 = Graph([Graph([inner_v2_mid, inner_v2_end], name='inner').as_node(name='nested'), finish_nested])

runner.run(outer_v1, {'x': 5}, workflow_id='wf-nested')
print('nested child run exists:', cp.get_run('wf-nested/nested') is not None)

try:
    runner.run(outer_v2, workflow_id='wf-nested')
except GraphChangedError as e:
    print('nested graph-change guard:', type(e).__name__)

nested_cp = cp.checkpoint('wf-nested')
nested_result = runner.run(outer_v2, {'x': 5}, checkpoint=nested_cp, workflow_id='wf-nested-v2')
print('nested fork id:', nested_result.workflow_id)
print('nested fork output:', nested_result['final_nested'])
print('nested child run exists after fork:', cp.get_run(f'{nested_result.workflow_id}/nested') is not None)

cp.lineage('wf-nested-v2')


## 8) PAUSED workflows (interrupt) require `AsyncRunner`

This is the only section here that uses `await`.


In [ ]:
await cp.initialize()
async_runner = AsyncRunner(checkpointer=cp)

@interrupt(output_name='decision')
def approval() -> str: ...

@node(output_name='final')
def finalize(decision: str) -> str:
    return f'approved={decision}'

paused_graph = Graph([approval, finalize])
paused = await async_runner.run(paused_graph, workflow_id='wf-paused')
print('paused status:', paused.status.value)
print('response key:', paused.pause.response_key)

resumed = await async_runner.run(
    paused_graph,
    {paused.pause.response_key: 'yes'},
    workflow_id='wf-paused',
)
print('resumed status:', resumed.status.value, '| final =', resumed['final'])


## 9) Query cheat-sheet (sync, no await)

Use these to inspect lineage quickly:
- `cp.runs(limit=...)`
- `cp.get_run(workflow_id)`
- `cp.steps(workflow_id)`
- `cp.state(workflow_id)`
- `cp.checkpoint(workflow_id)`
- `cp.lineage(workflow_id)`
- `cp.stats(workflow_id)`


In [ ]:
runs = cp.runs(limit=50)
summary = [
    {
        'id': r.id,
        'status': r.status.value,
        'forked_from': r.forked_from,
        'fork_superstep': r.fork_superstep,
    }
    for r in runs
]

summary[:15]


In [ ]:
# Optional cleanup
# await cp.close()
for p in [DB_PATH, Path(str(DB_PATH) + '-shm'), Path(str(DB_PATH) + '-wal')]:
    p.unlink(missing_ok=True)


In [ ]:
# Close checkpointer connection (recommended before reruns)
await cp.close()
